# LLM Classification Finetuning – Baseline Notebook

**Competition:** [LLM Classification Finetuning](https://www.kaggle.com/competitions/llm-classification-finetuning/overview)

**Goal:** Predict which of two LLM responses a human will prefer (or if it's a tie) given a conversation prompt.  
Output is a 3-class probability distribution: `[winner_model_a, winner_model_b, winner_tie]`.  
**Evaluation metric:** Log-loss (lower is better).

---

### Notebook structure
1. Setup & imports  
2. Load data & EDA  
3. Naive baseline (uniform / majority-class priors)  
4. DeBERTa-v3-small fine-tuning with LoRA (strong baseline)  
5. Generate submission  

## 1. Setup & Imports

In [ ]:
# Install/upgrade required libraries (run once on Kaggle)
# !pip install -q peft transformers accelerate bitsandbytes

In [ ]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_cosine_schedule_with_warmup,
)
from peft import (
    get_peft_model,
    LoraConfig,
    TaskType,
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

## 2. Load Data & EDA

In [ ]:
# Adjust path if running locally
DATA_DIR = Path("/kaggle/input/llm-classification-finetuning")

train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df  = pd.read_csv(DATA_DIR / "test.csv")
sub_df   = pd.read_csv(DATA_DIR / "sample_submission.csv")

print(f"Train shape : {train_df.shape}")
print(f"Test  shape : {test_df.shape}")
train_df.head(2)

In [ ]:
# Column overview
print("Train columns :", train_df.columns.tolist())
print("Test  columns :", test_df.columns.tolist())
train_df.dtypes

In [ ]:
# Label distribution
label_cols = ["winner_model_a", "winner_model_b", "winner_tie"]
label_counts = train_df[label_cols].sum()
label_counts.plot(kind="bar", title="Label distribution", color=["steelblue", "tomato", "goldenrod"])
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
print(label_counts / label_counts.sum())

In [ ]:
# Convert one-hot labels to integer class
# 0 = model_a wins, 1 = model_b wins, 2 = tie
train_df["label"] = (train_df[label_cols].values * [0, 1, 2]).sum(axis=1).astype(int)
print(train_df["label"].value_counts())

In [ ]:
# Text length analysis
for col in ["prompt", "response_a", "response_b"]:
    train_df[f"{col}_len"] = train_df[col].astype(str).str.len()

train_df[["prompt_len", "response_a_len", "response_b_len"]].describe()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["prompt_len", "response_a_len", "response_b_len"]):
    train_df[col].clip(upper=5000).hist(bins=50, ax=ax)
    ax.set_title(f"{col} distribution")
plt.tight_layout()
plt.show()

## 3. Naive Baseline

Predict the empirical prior probabilities for every test sample (no model needed).  
This gives the simplest possible log-loss floor to beat.

In [ ]:
prior = label_counts / label_counts.sum()
print("Class priors:")
print(prior.rename({"winner_model_a": "model_a", "winner_model_b": "model_b", "winner_tie": "tie"}))

# Evaluate on training data (optimistic estimate — no cross-validation)
y_true_oh = train_df[label_cols].values          # one-hot ground truth
naive_preds = np.tile(prior.values, (len(train_df), 1))  # same prior for all rows
naive_ll = log_loss(y_true_oh, naive_preds)
print(f"\nNaive prior log-loss (train): {naive_ll:.4f}")

In [ ]:
# Naive submission
naive_sub = sub_df.copy()
naive_sub[label_cols] = prior.values
naive_sub.to_csv("naive_submission.csv", index=False)
naive_sub.head()

## 4. DeBERTa-v3-small Fine-Tuning with LoRA

### Strategy
- Concatenate `[SEP] prompt [SEP] response_a [SEP] response_b` as the model input.
- Add a 3-class classification head on top of DeBERTa-v3-small.
- Apply LoRA to reduce trainable parameters (fits in a T4 GPU).
- Train with cross-entropy loss and cosine LR schedule.

In [ ]:
# ─── Config ───────────────────────────────────────────────────────────────────
MODEL_NAME   = "microsoft/deberta-v3-small"   # ~140M params; can swap for -base
MAX_LEN      = 512     # tokens per sample
BATCH_SIZE   = 8
GRAD_ACCUM   = 4       # effective batch = 32
EPOCHS       = 3
LR           = 2e-4
WARMUP_RATIO = 0.1
N_FOLDS      = 5
FOLD_IDX     = 0       # set to None to train on all data
NUM_LABELS   = 3

# LoRA hyper-parameters
LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def build_text(row):
    """Concatenate prompt + both responses into a single input string."""
    prompt   = str(row["prompt"])[:500]       # crude truncation before tokenising
    resp_a   = str(row["response_a"])[:1000]
    resp_b   = str(row["response_b"])[:1000]
    return f"{prompt} [SEP] {resp_a} [SEP] {resp_b}"


class PreferenceDataset(Dataset):
    def __init__(self, df, tokenizer, max_len, is_test=False):
        self.texts     = df.apply(build_text, axis=1).tolist()
        self.labels    = None if is_test else df["label"].tolist()
        self.tokenizer = tokenizer
        self.max_len   = max_len
        self.is_test   = is_test

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        if not self.is_test:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


print("Tokenizer loaded.")

In [ ]:
def build_model():
    base_model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_LABELS,
        ignore_mismatched_sizes=True,
    )

    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        target_modules=["query_proj", "key_proj", "value_proj", "dense"],
    )

    model = get_peft_model(base_model, lora_config)
    model.print_trainable_parameters()
    return model


def train_one_epoch(model, loader, optimizer, scheduler, scaler):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()

    for step, batch in enumerate(loader):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        with torch.cuda.amp.autocast():
            out  = model(**batch)
            loss = out.loss / GRAD_ACCUM

        scaler.scale(loss).backward()
        total_loss += out.loss.item()

        if (step + 1) % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

    return total_loss / len(loader)


@torch.no_grad()
def predict(model, loader):
    model.eval()
    all_probs = []
    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        logits = model(**batch).logits
        probs  = torch.softmax(logits, dim=-1).cpu().numpy()
        all_probs.append(probs)
    return np.concatenate(all_probs, axis=0)

In [ ]:
# ─── Cross-validation training loop ──────────────────────────────────────────
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

oof_preds  = np.zeros((len(train_df), NUM_LABELS))
test_preds = np.zeros((len(test_df),  NUM_LABELS))

test_dataset = PreferenceDataset(test_df, tokenizer, MAX_LEN, is_test=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE * 2, shuffle=False,
                          num_workers=2, pin_memory=True)

folds_to_run = [FOLD_IDX] if FOLD_IDX is not None else range(N_FOLDS)

for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df["label"])):
    if fold not in folds_to_run:
        continue
    print(f"\n{'='*60}")
    print(f" Fold {fold+1} / {N_FOLDS}")
    print(f"{'='*60}")

    tr_df  = train_df.iloc[train_idx].reset_index(drop=True)
    val_df = train_df.iloc[val_idx].reset_index(drop=True)

    tr_ds  = PreferenceDataset(tr_df,  tokenizer, MAX_LEN)
    val_ds = PreferenceDataset(val_df, tokenizer, MAX_LEN)

    tr_loader  = DataLoader(tr_ds,  batch_size=BATCH_SIZE, shuffle=True,
                            num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE * 2, shuffle=False,
                            num_workers=2, pin_memory=True)

    model     = build_model().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    total_steps   = (len(tr_loader) // GRAD_ACCUM) * EPOCHS
    warmup_steps  = int(total_steps * WARMUP_RATIO)
    scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    scaler    = torch.cuda.amp.GradScaler()

    best_ll   = np.inf
    best_ckpt = f"best_fold{fold}.pt"

    for epoch in range(EPOCHS):
        train_loss = train_one_epoch(model, tr_loader, optimizer, scheduler, scaler)
        val_probs  = predict(model, val_loader)
        val_oh     = val_df[label_cols].values
        val_ll     = log_loss(val_oh, val_probs)
        print(f"  Epoch {epoch+1}/{EPOCHS} | train_loss={train_loss:.4f} | val_log_loss={val_ll:.4f}")

        if val_ll < best_ll:
            best_ll = val_ll
            torch.save(model.state_dict(), best_ckpt)

    # Load best checkpoint for this fold
    model.load_state_dict(torch.load(best_ckpt, map_location=DEVICE))
    oof_preds[val_idx] = predict(model, val_loader)
    test_preds         += predict(model, test_loader) / len(folds_to_run)

    print(f"  Best val log-loss for fold {fold+1}: {best_ll:.4f}")

    del model
    torch.cuda.empty_cache()

In [ ]:
# ─── OOF score ───────────────────────────────────────────────────────────────
oof_mask = oof_preds.sum(axis=1) > 0   # only filled folds
oof_ll   = log_loss(train_df.loc[oof_mask, label_cols].values, oof_preds[oof_mask])
print(f"OOF Log-Loss: {oof_ll:.4f}")

## 5. Generate Submission

In [ ]:
submission = sub_df.copy()
submission[label_cols] = test_preds

# Sanity checks
assert np.allclose(submission[label_cols].sum(axis=1), 1.0, atol=1e-5), \
    "Probabilities should sum to 1!"
assert (submission[label_cols] >= 0).all().all(), \
    "Probabilities should be non-negative!"

submission.to_csv("submission.csv", index=False)
print("Submission saved!")
submission.head()

In [ ]:
# Quick look at predicted distribution
pred_df = pd.DataFrame(test_preds, columns=label_cols)
pred_df.describe()

---
## Summary & Next Steps

| Approach | Expected LB log-loss |
|---|---|
| Naive prior | ~1.05 |
| DeBERTa-v3-small + LoRA (this notebook) | ~0.90–0.95 |
| DeBERTa-v3-base / large + LoRA | ~0.85–0.90 |
| Ensemble of large LLMs | < 0.85 |

### Ideas to improve
1. **Larger backbone** – swap `deberta-v3-small` for `deberta-v3-base` or `deberta-v3-large`.  
2. **Better text construction** – use the full conversation history if available; try different separator strategies.  
3. **Label smoothing** – helps with over-confident predictions.  
4. **Longer context** – increase `MAX_LEN` to 1024 / 2048 with a model that supports it.  
5. **Ensemble** – average predictions from multiple folds or multiple model families.  
6. **Reward model** – fine-tune an open-source reward model (e.g., `OpenAssistant/reward-model-deberta-v3-large-v2`).  
7. **Positional swap augmentation** – randomly swap response_a and response_b during training to mitigate position bias.